# TP - Deep Learning avancé - attaques adverses

Le but de TP est d'implémenter l'attaque adverse d'un réseau CNN appris par ailleurs.

## Chargement des données

Pour ce TP, nous allons utiliser à nouveau utiliser le dataset MNIST (la drosophile du machine learing) pour faire un modèle de classification d'images. Nous allons donc, dans un premier temps, téléchargez le dataset et le préparer pour l'entraînement.

In [2]:
# Chargement de MNIST

from torchvision import datasets, transforms

# Chargement des données
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)


100.0%
100.0%
100.0%
100.0%


In [3]:
#Créez vos dataloaders (attention à la taille des batchs !)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

## Création d'un réseau CNN

En reprenant le code du TP précédent, on crée un modèle CNN pour la classification des images de MNIST.

In [11]:
# Definition du modele
import torch.nn as nn
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1,padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1,padding=1)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self,x):
        x = torch.relu(torch.max_pool2d(self.conv1(x),2))
        x = torch.relu(torch.max_pool2d(self.conv2(x),2))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x


Implémentez votre processus d'apprentissage sur 1 epoch avec un learning rate de 10e-3 (en utilisant l'optimiseur Adam) et une fonction de coût de type CrossEntropyLoss.

Vérifiez le taux de bonne classifcation en test de ce simple modèle.

In [5]:
# -------------------------------
# Train the Model
# -------------------------------
def train(model, loader, optimizer, epochs=15):
    model.train()
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        for data, target in loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} completed")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallNet().to(device)

In [16]:
import torch
model = CNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

In [17]:
train(model, train_loader, optimizer, epochs=1)

## Attaque adverse

Instancier la fonction pgd_attack, qui étant donné un modèle, une image et son label, va produire une image corrompue telle que

$\max_e \mathcal{L}(x+\eta, y)$

tel que $x+\eta \in [0,1]^{n\times n}$ et $||\eta||_∞ \leqϵ$


Pour cela, on va faire une descente de gradient projeté à pas fixe (avec $\alpha = 0.01$ par défaut) et une nombre maximum d'itération.

On fera pour une nombre d'itération donné
- calcule du gradient $∇_x\mathcal{L}(x,y)$ et mise à jour $\eta ← \alpha * ∇_x\mathcal{L}(x,y)$. En pratique, on utilisera le signe du gradient.
- vérification de la contrainte $||\eta||_∞ \leq ϵ$ (on utilisera torch.clamp pour que $∀ i, -ϵ\leq \eta_i \leq ϵ$)
- image adversariale $ x'← x + \eta$
- projection de $x'$ sur l'espace des solution admissible (on utilisera à nouveau torch.clamp pour que $\forall (i,j), x'_{ij} \in [0, 1] $)

NB :
* pour notre cas, $\mathcal{L}$ sera l'entropie croisée, on cherchera une perburbation qui change le label (en pratique, le classifieur ira au moindre effort et essayera de "pousser" vers la deuxième classe la plus probable).
* on est sur une attaque non specifiée (on trompe le classifieur mais pas vers une classe particulière) mais on pourrait forcer le classifieur vers une classe définie

In [66]:
# attaque PGD
def pgd_attack(model, images, labels, epsilon=0.3, alpha=0.01, num_iter=30):
    images = images.clone().detach().to(torch.float).requires_grad_(True)
    original_images = images.data

    for _ in range(num_iter):
        outputs = model(images)
        model.zero_grad()
        loss = nn.cross_entropy(outputs, labels)
        loss.backward()

        # PGD update step
        adv_images = images + alpha * images.grad.sign()
        eta = torch.clamp(adv_images - original_images, min=-epsilon, max=epsilon)
        images = torch.clamp(original_images + eta, min=0, max=1).detach_()
        images.requires_grad = True

    return images

In [67]:
# Function to show images
def imshow(img, title=None):
    img = img.detach().squeeze().cpu().numpy()  # Detach from graph, then convert to NumPy array
    plt.imshow(img, cmap='gray')  # MNIST is grayscale, so we use cmap='gray'
    if title is not None:
        plt.title(title)
    plt.axis('off')

def show_original_and_perturbed(model, test_loader, epsilon=0.3):
    model.eval()
    for data, target in test_loader:
        # Get the first image and label in the batch
        original_img = data[0].unsqueeze(0)  # Take the first image and add a batch dimension
        label = target[0]

        # Create adversarial example using PGD
        adv_img = pgd_attack(model, original_img, label.unsqueeze(0), epsilon=epsilon)


        output = model(original_img)
        _, origin_pred = torch.max(output.data, 1)


        output = model(adv_img)
        _, adv_pred = torch.max(output.data, 1)

        print('Original Prediction:', origin_pred.item(), ' Perturbed Prediction:', adv_pred.item(), 'Label:', label.item())

        # Plot original and perturbed images side by side
        plt.figure(figsize=(6, 3))

        # Show original image
        plt.subplot(1, 2, 1)
        imshow(original_img.squeeze(), title='Original Image')  # Squeeze to remove unnecessary dimensions

        # Show perturbed image
        plt.subplot(1, 2, 2)
        imshow(adv_img.squeeze(), title=f'Perturbed Image (ε={epsilon})')

        plt.show()
        break  # We only need to show the first example


In [68]:
show_original_and_perturbed(model, test_loader, epsilon=0.3)


AttributeError: module 'torch.nn' has no attribute 'cross_entropy'

In [ ]:
# Evaluation du modele sous attaque PGD
def evaluate_under_attack(model, test_loader, epsilon=0.3):
    model.eval()
    correct = 0
    total = 0
    for data, target in test_loader:
        adv_data = pgd_attack(model, data, target, epsilon=epsilon)
        output = model(adv_data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

    print(f'Accuracy sous attaque PGD (ε={epsilon}): {100 * correct / total:.2f}%')

In [ ]:
# evaluation du modele en pratique
evaluate_under_attack(model, test_loader, epsilon=0.3)

## Visualisation

- Prendre la première image de l'ensemble de test et l'afficher (en utilisant imshow de Matplotlib) et comparer la prédiction par le CCN de son label avec la vérité terrain.
- Générer une image adversarial à partir de cette image, l'afficher et comparer la prédiction du CNN sur cet exemple.

NB
* la norme $\ell_∞$ utilisée rend l'attaque peu discrète, une norme  $\ell_1$ ou $\ell_0$ serait plus furtive. On pourrait aussi considérer $\ell_2$.
* malgré l'attaque, l'exemple reste facilement identifiable pour un humain.

In [ ]:
import matplotlib.pyplot as plt



## pour aller plus loin
- tester la sensibilité de l'attaque aux différents paramètres ($\alpha$, $\epsilon$, nombre d'itérations... )
- instantier une autre attaque FGSM ou Carlini and Wagner (C&W)
- inclure cette attaque pour faire de l'adversarial training (on générera de nouveaux exemples adversariaux à chaque epoch)
- attaquer un réseau pré-appris (en téléchargeant les poids d'un CNN LeNet pour ImageNet).
- lire le papier "EXPLAINING AND HARNESSING
ADVERSARIAL EXAMPLES" (Goodfellow et al. ICLR 2015) https://arxiv.org/pdf/1412.6572
- lire le papier "Towards Deep Learning Models Resistant to Adversarial Attacks" (Madry et al. ICLR 2018) https://openreview.net/pdf?id=rJzIBfZAb